# CS336 Spring 2026 - File-based `train_bpe()` + 多进程 Pre-tokenization
----
本节目标只有一个：
```text
input_path → binary file → safe chunk boundaries
→ local pre-token Counter → Counter aggregation
→ PretokenCounts → BPE core → vocab + ordered merges
```

核心不是“怎么开进程”，而是先保证：

$$
C_{\text{whole}}=\sum_i C_{\text{safe chunk }i}
$$

任意切块并不满足这个等式。

In [2]:
from __future__ import annotations

from collections import Counter
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path
from time import perf_counter
import io
import os
import random
import tempfile
import regex

TokenSequence = tuple[bytes, ...]
PretokenCounts = Counter[TokenSequence]
Pair = tuple[bytes, bytes]

GPT2_PRETOKEN_PATTERN = (
    r"'(?:[sdmt]|ll|ve|re)"
    r"| ?\p{L}+"
    r"| ?\p{N}+"
    r"| ?[^\s\p{L}\p{N}]+"
    r"|\s+(?!\S)"
    r"|\s+"
)


def pretokenize(text: str) -> list[str]:
    return [m.group(0) for m in regex.finditer(GPT2_PRETOKEN_PATTERN, text)]


assert "".join(pretokenize("Hello 世界 🌍!")) == "Hello 世界 🌍!"
print("环境和 regex pre-tokenization: 通过")

环境和 regex pre-tokenization: 通过


## 1. 本节的困难
> 本节真正的困难是: 文件应该在哪切?

假设完整文本为 $T$, 切成
$$
T=T_1+T_2+\cdots+T_k
$$
希望有
$$
C(T)=\sum_{i=1}^k C(T_i)
$$
其中 $C$ 表示 pre-tokenization+frequency counting.
> 但是任意切分文件则会破坏这个条件, 因此后续要解决此问题.


例如, 完整的 `" wonderful"` 是一个 pre-token, 但要是切成 `" won"`和 `"derful"` 就变成了两个. 因此必须分布验证:
- UTF-8 正确性: 每块能解码, 并且重组时字节不丢失不重复
- pre-tokenization 语义正确性: 分块后 Counter 与完整处理相同.

In [3]:
whole = " wonderful"
arbitrary_parts = [" won", "derful"]
print("完整:", pretokenize(whole))
print("任意分块:", [part for text in arbitrary_parts for part in pretokenize(text)])
assert pretokenize(whole) != [
    part for text in arbitrary_parts for part in pretokenize(text)
]
print("UTF-8字节:", list("你".encode("utf-8")))
try:
    "你".encode("utf-8")[:1].decode("utf-8")
except UnicodeDecodeError:
    print("UTF-8 半字符边界：正确地触发了 UnicodeDecodeError")


完整: [' wonderful']
任意分块: [' won', 'derful']
UTF-8字节: [228, 189, 160]
UTF-8 半字符边界：正确地触发了 UnicodeDecodeError


## 2. 安全边界
本节利用 `<|endoftext|>` 作为 document delimiter(文档分隔符). 因为 special token 在普通 BPE 统计前就被隔离, 不应该跨越该边界进行 BPE merge.

官方实例的思路大概是:
```text
文件大小 → 等距估计 byte offsets
→ 从每个内部 offset 向后找 delimiter
→ 移到 delimiter 起点
→ sorted(set(boundaries))
```

多个估计位置可能落到同一个 delimiter, 所以实际 chunk 数可能少于请求数.

In [5]:
def find_chunk_boundaries(
    file,
    desired_num_chunks: int,
    split_special_token: bytes,
    *,
    read_ahead: int = 4096,
) -> list[int]:
    """在不切断 document delimiter 的位置建立 byte boundaries。"""
    if desired_num_chunks < 1:
        raise ValueError("desired_num_chunks must be >= 1")
    if not isinstance(split_special_token, bytes) or not split_special_token:
        raise ValueError("split_special_token must be nonempty bytes")
    if read_ahead < 1:
        raise ValueError("read_ahead must be >= 1")

    file.seek(0, os.SEEK_END)
    file_size = file.tell()
    file.seek(0)

    approx_size = file_size // desired_num_chunks
    boundaries = [i * approx_size for i in range(desired_num_chunks + 1)]
    boundaries[-1] = file_size

    keep = len(split_special_token) - 1

    for i in range(1, len(boundaries) - 1):
        position = boundaries[i]
        file.seek(position)
        tail = b""

        while True:
            block = file.read(read_ahead)
            if not block:
                boundaries[i] = file_size
                break

            searchable = tail + block
            found = searchable.find(split_special_token)
            if found >= 0:
                boundaries[i] = position - len(tail) + found
                break

            tail = searchable[-keep:] if keep else b""
            position += len(block)

    return sorted(set(boundaries))


## 3. 故意将 delimiter 放在读取块边界
测试时将 `read_ahead` 缩小到5bytes, 但是 `<|endoftext|>` 本身有13bytes. 这样它必然可能横跨多个读取块, 可以用来测试程序.
- `b"".join(chunks) == original`: 不能丢失或重复字节
- 每个 chunk 能完整的 UTF-8 decode
- 内部 boundary 对应 delimiter 的起点

In [6]:
DELIMITER = b"<|endoftext|>"
raw = ("开头" + "<|endoftext|>" + "alpha 🌍 " + "<|endoftext|>" + "尾部").encode(
    "utf-8"
)

with io.BytesIO(raw) as file:
    boundaries = find_chunk_boundaries(file, 7, DELIMITER, read_ahead=5)

pieces = [raw[start:end] for start, end in zip(boundaries[:-1], boundaries[1:])]

assert b"".join(pieces) == raw
assert all(piece.decode("utf-8") is not None for piece in pieces)
assert all(raw[b : b + len(DELIMITER)] == DELIMITER for b in boundaries[1:-1])

print("boundary 数:", len(boundaries))
print("实际 chunk 数:", len(pieces))
print("跨 read() 块的 delimiter 检查: 通过")
print("完整重组 + UTF-8 decode: 通过")

boundary 数: 3
实际 chunk 数: 2
跨 read() 块的 delimiter 检查: 通过
完整重组 + UTF-8 decode: 通过


## 4. 测试矩阵：不要只测 happy path

检查以下情况：空文件、没有分隔符、连续分隔符、Unicode / emoji、delimiter 横跨读取缓冲区、期望 chunk 数远大于文档数。

**核心不变量：**

- boundaries 非递减、范围合法；
- chunks 拼接与原始 bytes 严格相等；
- 非空块能严格 UTF-8 decode；
- 内部 boundary 必须指向 delimiter 开始。


In [7]:
examples = [
    b"",
    b"just ordinary bytes",
    b"<|endoftext|><|endoftext|>",
    "第一篇🌍<|endoftext|>第二篇你好<|endoftext|>第三篇".encode("utf-8"),
]

for raw_case in examples:
    with io.BytesIO(raw_case) as f:
        cuts = find_chunk_boundaries(
            f, desired_num_chunks=11, split_special_token=b"<|endoftext|>", read_ahead=2
        )
    slices = [raw_case[a:b] for a, b in zip(cuts, cuts[1:])]

    assert cuts == sorted(set(cuts))
    assert b"".join(slices) == raw_case
    assert all(raw_case[i : i + 13] == b"<|endoftext|>" for i in cuts[1:-1])
    assert all(piece.decode("utf-8") is not None for piece in slices)

    print("byte length =", len(raw_case), "; boundaries =", cuts)


byte length = 0 ; boundaries = [0]
byte length = 19 ; boundaries = [0, 19]
byte length = 26 ; boundaries = [0, 13, 26]
byte length = 63 ; boundaries = [0, 13, 41, 63]


## 5. Math：什么时候可以使用 `Counter` aggregation？

令 `D(text)` 表示按**特殊标记**隔离后的 ordinary spans，`P(span)` 表示 regex pre-token 序列。

在切块边界也是合法文档边界的条件下，计数满足：

$$
C_{\text{corpus}}(p)
= \sum_{\text{ordinary span } s} \#\{p \in P(s)\}
= \sum_{\text{chunk } c} C_c(p)
$$

这里的 `p` 是普通 pre-token，**不包括 special token 本身**。这才是 `Counter` 的局部可加性；并不是说 regex 可以对任何随机切块独立处理。


In [8]:
delimiter_text = "<|endoftext|>"
corpus = "你好 world<|endoftext|> wonderful cat<|endoftext|>🌍 你好"
corpus_bytes = corpus.encode("utf-8")


def ordinary_pretoken_counts(text):
    # 学习用的计数观察：不生成 merges、vocab，也不实现官方 train_bpe。
    counts = Counter()
    for ordinary_span in text.split(delimiter_text):
        counts.update(pretokenize(ordinary_span))
    return counts


whole_counts = ordinary_pretoken_counts(corpus)

with io.BytesIO(corpus_bytes) as file:
    offsets = find_chunk_boundaries(file, 8, delimiter_text.encode(), read_ahead=3)
chunks = [corpus_bytes[a:b].decode("utf-8") for a, b in zip(offsets, offsets[1:])]
local_counts = [ordinary_pretoken_counts(chunk) for chunk in chunks]
merged_counts = sum(local_counts, Counter())

assert merged_counts == whole_counts
print("安全切块:", offsets)
print("局部统计:", local_counts)
print("一致性:", merged_counts == whole_counts)


安全切块: [0, 12, 39, 63]
局部统计: [Counter({'你好': 1, ' world': 1}), Counter({' wonderful': 1, ' cat': 1}), Counter({'🌍': 1, ' 你好': 1})]
一致性: True
